# Análise de Dados com SQL

https://deepnote.com/app/infnet-1d7e/SQL-1d66470e-937e-4aac-9f14-1f15a78e4843?utm_content=1d66470e-937e-4aac-9f14-1f15a78e4843&__run=true

### Query 1.1 — Ranking de produtos por lucro usando RANK()

In [43]:
df_1 = _dntk.execute_sql(
  '-- Classific produtos dentro de cada categoria com base no lucro total.\n-- Usa RANK() para permitir empates no ranking.\nSELECT\n    "Category",\n    "Sub-Category",\n    "Product Name",\n    ROUND(SUM("Profit"), 2) AS total_profit,\n    RANK() OVER (\n        PARTITION BY "Category"        -- Divide o ranking por categoria\n        ORDER BY SUM("Profit") DESC    -- Ordena pelo maior lucro\n    ) AS profit_rank\nFROM "superstoreutf8_.csv"\nGROUP BY\n    "Category",\n    "Sub-Category",\n    "Product Name"\nORDER BY\n    "Category",\n    profit_rank;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_1

,Category,Sub-Category,Product Name,total_profit,profit_rank
0,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",1927.44,1
1,Furniture,Chairs,Global Deluxe High-Back Manager's Chair,1558.59,2
2,Furniture,Chairs,Hon Pagoda Stacking Chairs,1540.70,3
3,Furniture,Chairs,Hon 4070 Series Pagoda Armless Upholstered Sta...,1388.63,4
4,Furniture,Chairs,Office Star - Professional Matrix Back Chair w...,1305.65,5
...,...,...,...,...,...
1845,Technology,Machines,Epson TM-T88V Direct Thermal Printer - Monochr...,-1057.23,408
1846,Technology,Machines,Cisco TelePresence System EX90 Videoconferenci...,-1811.08,409
1847,Technology,Machines,Cubify CubeX 3D Printer Triple Head Print,-3839.99,410
1848,Technology,Machines,Lexmark MX611dhe Monochrome Laser Printer,-4589.97,411


### Query 1.2 — Ranking por lucro com DENSE_RANK()

In [55]:
df_2 = _dntk.execute_sql(
  '-- Repete a mesma análise, mas agora usando DENSE_RANK().\n-- Diferença: DENSE_RANK() não pula posições em caso de empate.\nSELECT\n    "Category",\n    "Sub-Category",\n    "Product Name",\n    ROUND(SUM("Profit"), 2) AS total_profit,\n    DENSE_RANK() OVER (\n        PARTITION BY "Category"\n        ORDER BY SUM("Profit") DESC\n    ) AS profit_dense_rank\nFROM "superstoreutf8_.csv"\nGROUP BY\n    "Category",\n    "Sub-Category",\n    "Product Name"\nORDER BY\n    "Category",\n    profit_dense_rank;\n\n-- Diferença entre RANK() e DENSE_RANK():\n-- Se dois produtos empatarem em 1º lugar:\n--   - RANK() -> o próximo será classificado como 3º.\n--   - DENSE_RANK() -> o próximo será classificado como 2º.\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_2

,Category,Sub-Category,Product Name,total_profit,profit_dense_rank
0,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",1927.44,1
1,Furniture,Chairs,Global Deluxe High-Back Manager's Chair,1558.59,2
2,Furniture,Chairs,Hon Pagoda Stacking Chairs,1540.70,3
3,Furniture,Chairs,Hon 4070 Series Pagoda Armless Upholstered Sta...,1388.63,4
4,Furniture,Chairs,Office Star - Professional Matrix Back Chair w...,1305.65,5
...,...,...,...,...,...
1845,Technology,Machines,Epson TM-T88V Direct Thermal Printer - Monochr...,-1057.23,408
1846,Technology,Machines,Cisco TelePresence System EX90 Videoconferenci...,-1811.08,409
1847,Technology,Machines,Cubify CubeX 3D Printer Triple Head Print,-3839.99,410
1848,Technology,Machines,Lexmark MX611dhe Monochrome Laser Printer,-4589.97,411


### Query 1.3 — Ranking por quantidade de vendas por cliente usando ROW_NUMBER()

In [58]:
df_3 = _dntk.execute_sql(
  '-- Gera um ranking sequencial por cliente, dentro de cada segmento,\n-- baseado na quantidade de vendas (Order Quantity).\nSELECT\n    "Segment",\n    "Customer ID",\n    "Customer Name",\n    SUM("Quantity") AS total_quantity,\n    ROW_NUMBER() OVER (\n        PARTITION BY "Segment"        -- Ranking separado por segmento\n        ORDER BY SUM("Quantity") DESC -- Clientes com mais vendas primeiro\n    ) AS quantity_row_number\nFROM "superstoreutf8_.csv"\nGROUP BY\n    "Segment",\n    "Customer ID",\n    "Customer Name"\nORDER BY\n    "Segment",\n    quantity_row_number;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_3

,Segment,Customer ID,Customer Name,total_quantity,quantity_row_number
0,Consumer,WB-21850,William Brown,146.0,1
1,Consumer,JL-15835,John Lee,143.0,2
2,Consumer,SC-20725,Steven Cartwright,133.0,3
3,Consumer,EP-13915,Emily Phan,124.0,4
4,Consumer,CB-12025,Cassandra Brandow,122.0,5
...,...,...,...,...,...
788,Home Office,MZ-17335,Maria Zettner,11.0,144
789,Home Office,RS-19870,Roy Skaria,10.0,145
790,Home Office,AS-10135,Adrian Shami,9.0,146
791,Home Office,EL-13735,Ed Ludwig,9.0,147


### Query 2.1 — Diferença de lucro entre pedidos usando LAG()

In [61]:
df_4 = _dntk.execute_sql(
  '-- Calcula a diferença de lucro entre um pedido e o anterior do mesmo cliente.\n-- Ordena por Customer ID e Data do Pedido.\nSELECT\n    "Customer ID",\n    "Customer Name",\n    "Order Date",\n    SUM("Profit") AS current_profit,\n    LAG(SUM("Profit")) OVER (\n        PARTITION BY "Customer ID"         -- Agrupa por cliente\n        ORDER BY "Order Date"              -- Ordena cronologicamente\n    ) AS previous_profit,\n    SUM("Profit") -\n    LAG(SUM("Profit")) OVER (\n        PARTITION BY "Customer ID"\n        ORDER BY "Order Date"\n    ) AS profit_difference\nFROM "superstoreutf8_.csv"\nGROUP BY\n    "Customer ID",\n    "Customer Name",\n    "Order Date"\nORDER BY\n    "Customer ID",\n    "Order Date";\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_4

,Customer ID,Customer Name,Order Date,current_profit,previous_profit,profit_difference
0,AA-10315,Alex Avila,2014-03-31,267.4224,NaN,NaN
1,AA-10315,Alex Avila,2014-09-15,13.2826,267.4224,-254.1398
2,AA-10315,Alex Avila,2015-10-04,7.0096,13.2826,-6.2730
3,AA-10315,Alex Avila,2016-03-03,-747.1021,7.0096,-754.1117
4,AA-10315,Alex Avila,2017-06-29,96.5050,-747.1021,843.6071
...,...,...,...,...,...,...
4987,ZD-21925,Zuschuss Donatelli,2014-08-27,25.8774,NaN,NaN
4988,ZD-21925,Zuschuss Donatelli,2016-04-03,146.8280,25.8774,120.9506
4989,ZD-21925,Zuschuss Donatelli,2016-05-05,3.3440,146.8280,-143.4840
4990,ZD-21925,Zuschuss Donatelli,2016-07-08,56.4925,3.3440,53.1485


### Query 2.2 — Diferença de quantidade usando LEAD()

In [64]:
df_5 = _dntk.execute_sql(
  '-- Calcula a diferença na quantidade de itens entre um pedido e o próximo do mesmo cliente.\nSELECT\n    "Customer ID",\n    "Customer Name",\n    "Order Date",\n    SUM("Quantity") AS current_quantity,\n    LEAD(SUM("Quantity")) OVER (\n        PARTITION BY "Customer ID"         -- Agrupa por cliente\n        ORDER BY "Order Date"              -- Ordena por data do pedido\n    ) AS next_quantity,\n    LEAD(SUM("Quantity")) OVER (\n        PARTITION BY "Customer ID"\n        ORDER BY "Order Date"\n    ) - SUM("Quantity") AS quantity_difference\nFROM "superstoreutf8_.csv"\nGROUP BY\n    "Customer ID",\n    "Customer Name",\n    "Order Date"\nORDER BY\n    "Customer ID",\n    "Order Date";\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_5

,Customer ID,Customer Name,Order Date,current_quantity,next_quantity,quantity_difference
0,AA-10315,Alex Avila,2014-03-31,4.0,5.0,1.0
1,AA-10315,Alex Avila,2014-09-15,5.0,2.0,-3.0
2,AA-10315,Alex Avila,2015-10-04,2.0,14.0,12.0
3,AA-10315,Alex Avila,2016-03-03,14.0,5.0,-9.0
4,AA-10315,Alex Avila,2017-06-29,5.0,NaN,NaN
...,...,...,...,...,...,...
4987,ZD-21925,Zuschuss Donatelli,2014-08-27,9.0,8.0,-1.0
4988,ZD-21925,Zuschuss Donatelli,2016-04-03,8.0,5.0,-3.0
4989,ZD-21925,Zuschuss Donatelli,2016-05-05,5.0,7.0,2.0
4990,ZD-21925,Zuschuss Donatelli,2016-07-08,7.0,3.0,-4.0


### Query 2.3 — Identificar clientes com flutuações altas (> 500) usando LAG() e LEAD()

In [70]:
df_6 = _dntk.execute_sql(
  '-- Identifica clientes com variações de vendas maiores que 500 usando LAG e LEAD\nWITH vendas_variacoes AS (\n    SELECT\n        "Customer ID",\n        "Customer Name",\n        "Order Date",\n        SUM("Sales") AS current_sales,\n        LAG(SUM("Sales")) OVER (\n            PARTITION BY "Customer ID"\n            ORDER BY "Order Date"\n        ) AS previous_sales,\n        LEAD(SUM("Sales")) OVER (\n            PARTITION BY "Customer ID"\n            ORDER BY "Order Date"\n        ) AS next_sales\n    FROM "superstoreutf8_.csv"\n    GROUP BY\n        "Customer ID",\n        "Customer Name",\n        "Order Date"\n)\nSELECT\n    "Customer ID",\n    "Customer Name",\n    "Order Date",\n    current_sales,\n    previous_sales,\n    next_sales,\n    ABS(current_sales - COALESCE(previous_sales, 0)) AS diff_previous,\n    ABS(COALESCE(next_sales, 0) - current_sales) AS diff_next\nFROM vendas_variacoes\nWHERE\n    ABS(current_sales - COALESCE(previous_sales, 0)) > 500\n    OR\n    ABS(COALESCE(next_sales, 0) - current_sales) > 500\nORDER BY\n    diff_previous DESC,\n    diff_next DESC;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_6

,Customer ID,Customer Name,Order Date,current_sales,previous_sales,next_sales,diff_previous,diff_next
0,SM-20320,Sean Miller,2014-03-18,23661.228,NaN,837.444,23661.228,22823.784
1,SM-20320,Sean Miller,2015-08-21,837.444,23661.228,9.960,22823.784,827.484
2,TC-20980,Tamara Chand,2016-11-26,7.312,18336.740,NaN,18329.428,7.312
3,TC-20980,Tamara Chand,2016-10-02,18336.740,85.848,7.312,18250.892,18329.428
4,RB-19360,Raymond Buch,2017-07-25,20.230,14052.480,130.568,14032.250,110.338
...,...,...,...,...,...,...,...,...
2378,ES-14080,Erin Smith,2015-02-27,599.900,598.144,37.940,1.756,561.960
2379,SL-20155,Sara Luxemburg,2014-10-03,67.760,66.030,1166.920,1.730,1099.160
2380,SG-20470,Sheri Gordon,2016-08-22,26.352,26.820,1044.630,0.468,1018.278
2381,SH-19975,Sally Hughsby,2017-10-27,24.288,24.620,627.840,0.332,603.552


### Query 3.1 — Média móvel de lucro por estado

In [76]:
df_7 = _dntk.execute_sql(
  '-- Calcula a média móvel de lucro por estado, considerando os últimos 3 pedidos\nWITH lucro_por_pedido AS (\n    SELECT\n        "State",\n        "Order Date",\n        SUM("Profit") AS total_profit\n    FROM "superstoreutf8_.csv"\n    GROUP BY "State", "Order Date"\n)\nSELECT\n    "State",\n    "Order Date",\n    total_profit,\n    AVG(total_profit) OVER (\n        PARTITION BY "State"\n        ORDER BY "Order Date"\n        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW -- Janela de 3 pedidos: atual + 2 anteriores\n    ) AS moving_avg_profit\nFROM lucro_por_pedido\nORDER BY "State", "Order Date";\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_7

,State,Order Date,total_profit,moving_avg_profit
0,Alabama,2014-04-07,2.7776,2.777600
1,Alabama,2014-04-08,316.1392,159.458400
2,Alabama,2014-05-22,46.5810,121.832600
3,Alabama,2014-10-18,444.6902,269.136800
4,Alabama,2014-11-28,3.9609,165.077367
...,...,...,...,...
4205,Wisconsin,2017-11-27,75.3001,41.876700
4206,Wisconsin,2017-12-08,419.3294,178.795567
4207,Wisconsin,2017-12-17,8.4656,167.698367
4208,Wisconsin,2017-12-18,33.8443,153.879767


### Query 3.2 — Soma das vendas do cliente nas 4 ordens anteriores

In [79]:
df_8 = _dntk.execute_sql(
  '-- Para cada peddo, soma o total de vendas do cliente nas 4 ordens anteriores\nWITH vendas_por_pedido AS (\n    SELECT\n        "Customer ID",\n        "Order Date",\n        SUM("Sales") AS total_sales\n    FROM "superstoreutf8_.csv"\n    GROUP BY "Customer ID", "Order Date"\n)\nSELECT\n    "Customer ID",\n    "Order Date",\n    total_sales,\n    SUM(total_sales) OVER (\n        PARTITION BY "Customer ID"\n        ORDER BY "Order Date"\n        ROWS BETWEEN 4 PRECEDING AND 1 PRECEDING -- 4 pedidos anteriores, excluindo o atual\n    ) AS sum_last_4_orders\nFROM vendas_por_pedido\nORDER BY "Customer ID", "Order Date";\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_8

,Customer ID,Order Date,total_sales,sum_last_4_orders
0,AA-10315,2014-03-31,726.548,NaN
1,AA-10315,2014-09-15,29.500,726.548
2,AA-10315,2015-10-04,26.960,756.048
3,AA-10315,2016-03-03,4406.072,783.008
4,AA-10315,2017-06-29,374.480,5189.080
...,...,...,...,...
4987,ZD-21925,2014-08-27,244.760,NaN
4988,ZD-21925,2016-04-03,331.080,244.760
4989,ZD-21925,2016-05-05,16.720,575.840
4990,ZD-21925,2016-07-08,839.944,592.560


### Query 3.3 — Número acumulado de pedidos por subcategoria

In [82]:
df_9 = _dntk.execute_sql(
  '-- Conta o número acumulado de pedidos por subcategoria\nWITH pedidos_por_data AS (\n    SELECT\n        "Sub-Category",\n        "Order Date",\n        COUNT(DISTINCT "Order ID") AS pedidos_dia\n    FROM "superstoreutf8_.csv"\n    GROUP BY "Sub-Category", "Order Date"\n)\nSELECT\n    "Sub-Category",\n    "Order Date",\n    pedidos_dia,\n    COUNT(pedidos_dia) OVER (\n        PARTITION BY "Sub-Category"\n        ORDER BY "Order Date"\n        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW -- Acumulado desde o início\n    ) AS cumulative_orders\nFROM pedidos_por_data\nORDER BY "Sub-Category", "Order Date";\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_9

,Sub-Category,Order Date,pedidos_dia,cumulative_orders
0,Accessories,2014-01-09,1,1
1,Accessories,2014-01-13,1,2
2,Accessories,2014-01-15,1,3
3,Accessories,2014-02-01,1,4
4,Accessories,2014-02-07,1,5
...,...,...,...,...
6558,Tables,2017-12-10,1,253
6559,Tables,2017-12-11,1,254
6560,Tables,2017-12-14,2,255
6561,Tables,2017-12-22,4,256


### Query 4 — Reutilização de janela com WINDOW

In [85]:
df_10 = _dntk.execute_sql(
  '-- Define uma janela única para análises por região e categoria,\n-- ordena produtos por vendas decrescentes.\n-- Depois, reutiliza essa janela para calcular RANK, ROW_NUMBER e AVG.\n\nSELECT\n    "Region",\n    "Category",\n    "Product Name",\n    SUM("Sales") AS total_sales,\n    -- Ranking de produtos com base nas vendas\n    RANK() OVER w_region_produto AS sales_rank,\n    -- Número sequencial de cada produto dentro da região e categoria\n    ROW_NUMBER() OVER w_region_produto AS product_row_number,\n    -- Média de desconto por produto e região usando a mesma janela\n    AVG("Discount") OVER w_region_produto AS avg_discount\nFROM "superstoreutf8_.csv"\nGROUP BY\n    "Region",\n    "Category",\n    "Product Name",\n    "Discount"\nWINDOW w_region_produto AS (\n    PARTITION BY "Region", "Category"\n    ORDER BY SUM("Sales") DESC\n)\nORDER BY\n    "Region",\n    "Category",\n    sales_rank;\n\n\n-- Benefícios:\n    -- Menos código repetido\n    -- Mais fácil de ler\n    -- Mais fácil de manter\n    -- Mais performático → O mecanismo SQL calcula a janela uma única vez.',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_10

,Region,Category,Product Name,total_sales,sales_rank,product_row_number,avg_discount
0,Central,Furniture,HON 5400 Series Task Chairs for Big and Tall,3504.900,1,1,0.000000
1,Central,Furniture,HON 5400 Series Task Chairs for Big and Tall,3434.802,2,2,0.150000
2,Central,Furniture,"Atlantic Metals Mobile 5-Shelf Bookcases, Cust...",3069.996,3,3,0.206667
3,Central,Furniture,Office Star - Professional Matrix Back Chair w...,2807.840,4,4,0.155000
4,Central,Furniture,SAFCO Arco Folding Chair,2706.760,5,5,0.184000
...,...,...,...,...,...,...,...
6738,West,Technology,Texas Instrument TI-15 Fraction Calculator,11.560,340,340,0.145294
6739,West,Technology,"Verbatim Slim CD and DVD Storage Cases, 50/Pack",11.540,341,341,0.144868
6740,West,Technology,Maxell 4.7GB DVD-R 5/Pack,8.910,342,342,0.144444
6741,West,Technology,Kingston Digital DataTraveler 16GB USB 2.0,7.160,343,343,0.144606


### Query 5.1 — Ranking mensal de volume de vendas por região

In [88]:
df_11 = _dntk.execute_sql(
  '-- Ranking mensal de volume de vendas por região\nWITH vendas_mensais AS (\n    SELECT\n        "Region",\n        DATE_TRUNC(\'month\', "Order Date") AS order_month,\n        SUM("Sales") AS total_sales\n    FROM "superstoreutf8_.csv"\n    GROUP BY "Region", DATE_TRUNC(\'month\', "Order Date")\n)\nSELECT\n    "Region",\n    order_month,\n    total_sales,\n    RANK() OVER (\n        PARTITION BY order_month\n        ORDER BY total_sales DESC\n    ) AS regional_sales_rank\nFROM vendas_mensais\nORDER BY order_month, regional_sales_rank;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_11

,Region,order_month,total_sales,regional_sales_rank
0,South,2014-01-01,9322.0920,1
1,West,2014-01-01,2938.7230,2
2,Central,2014-01-01,1539.9060,3
3,East,2014-01-01,436.1740,4
4,South,2014-02-01,2028.9860,1
...,...,...,...,...
187,Central,2017-11-01,15154.9780,4
188,West,2017-12-01,29652.0950,1
189,East,2017-12-01,20084.4160,2
190,Central,2017-12-01,18883.0708,3


### Query 5.2 — Número de pedidos anteriores por cliente

In [91]:
df_12 = _dntk.execute_sql(
  '-- Número acumulado de pedidos por cliente até o pedido atual\nWITH pedidos_por_cliente AS (\n    SELECT\n        "Customer ID",\n        "Order Date",\n        COUNT(DISTINCT "Order ID") AS pedidos_dia\n    FROM "superstoreutf8_.csv"\n    GROUP BY "Customer ID", "Order Date"\n)\nSELECT\n    "Customer ID",\n    "Order Date",\n    pedidos_dia,\n    COUNT(pedidos_dia) OVER (\n        PARTITION BY "Customer ID"\n        ORDER BY "Order Date"\n        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW\n    ) AS total_pedidos_ate_hoje\nFROM pedidos_por_cliente\nORDER BY "Customer ID", "Order Date";\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_12

,Customer ID,Order Date,pedidos_dia,total_pedidos_ate_hoje
0,AA-10315,2014-03-31,1,1
1,AA-10315,2014-09-15,1,2
2,AA-10315,2015-10-04,1,3
3,AA-10315,2016-03-03,1,4
4,AA-10315,2017-06-29,1,5
...,...,...,...,...
4987,ZD-21925,2014-08-27,1,1
4988,ZD-21925,2016-04-03,1,2
4989,ZD-21925,2016-05-05,1,3
4990,ZD-21925,2016-07-08,1,4


### Query 5.3 — Top 5 clientes com maior crescimento de lucro mês a mês

In [94]:
df_13 = _dntk.execute_sql(
  '-- Top 5 clientes com maior crescimento de lucro mês a mês\nWITH lucro_mensal_cliente AS (\n    SELECT\n        "Customer ID",\n        "Customer Name",\n        DATE_TRUNC(\'month\', "Order Date") AS order_month,\n        SUM("Profit") AS total_profit\n    FROM "superstoreutf8_.csv"\n    GROUP BY "Customer ID", "Customer Name", DATE_TRUNC(\'month\', "Order Date")\n),\nvariacao_lucro AS (\n    SELECT\n        "Customer ID",\n        "Customer Name",\n        order_month,\n        total_profit,\n        LAG(total_profit) OVER (\n            PARTITION BY "Customer ID"\n            ORDER BY order_month\n        ) AS previous_profit,\n        total_profit - COALESCE(\n            LAG(total_profit) OVER (\n                PARTITION BY "Customer ID"\n                ORDER BY order_month\n            ), 0\n        ) AS profit_growth\n    FROM lucro_mensal_cliente\n)\nSELECT\n    "Customer ID",\n    "Customer Name",\n    order_month,\n    total_profit,\n    previous_profit,\n    profit_growth\nFROM variacao_lucro\nWHERE profit_growth > 0 -- Considera só crescimento, não queda\nORDER BY profit_growth DESC\nLIMIT 5;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_13

,Customer ID,Customer Name,order_month,total_profit,previous_profit,profit_growth
0,TC-20980,Tamara Chand,2016-10-01,8762.3891,37.7204,8724.6687
1,CS-12505,Cindy Stewart,2017-11-01,43.7060,-6886.5095,6930.2155
2,RB-19360,Raymond Buch,2017-03-01,6734.4720,72.6159,6661.8561
3,SC-20095,Sanjit Chand,2014-09-01,5511.8641,2.3328,5509.5313
4,AB-10105,Adrian Barton,2016-12-01,4946.3700,-204.4458,5150.8158


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=1d66470e-937e-4aac-9f14-1f15a78e4843' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>